# 31 — Attribution methods: who gets the credit?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/031-attribution-methods.ipynb)

Attribution methods answer a narrow question: how much of a score belongs to each input? Here you compute Shapley values exactly for a two-player game, average over every joining order, and then watch a LIME-style fit give a different answer every time you resample.

## 1. The numbers

Think of two features as two players who can team up. A value function says what any team is worth: alone, together, or empty. The goal is to divide the team's value between the players without inventing or losing any of it.

In [ ]:
import numpy as np                                      # the array library, preinstalled on Colab

v_empty = 0.0                                            # worth of the empty team
v1 = 0.6                                                 # worth of player 1 alone
v2 = 0.4                                                 # worth of player 2 alone
v_both = 1.0                                             # worth of both players together

print("v(empty)    =", v_empty)                          # 0.0, the baseline
print("v({1})      =", v1, "  v({2}) =", v2)              # 0.6 and 0.4
print("v({1,2})    =", v_both)                           # 1.0, more than either alone
print("total to share =", v_both - v_empty)               # 1.0 is the only number that must be split

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `v_empty` — the value of a team with nobody in it, the natural baseline.
- `v1` and `v2` — each feature's value when it acts alone.
- `v_both` — the value when both features are used.
- `v_both - v_empty` — the surplus that attribution has to divide.

## 2. Step by step: average the orderings

The Shapley trick is to imagine the players arriving one at a time. Each player's credit is what they add when they arrive, averaged over every possible arrival order. With two players there are only two orders, so the average is exact and short.

In [ ]:
v = {frozenset(): 0.0,                              # empty coalition
     frozenset({0}): 0.6,                              # only feature 1 joined
     frozenset({1}): 0.4,                              # only feature 2 joined
     frozenset({0, 1}): 1.0}                           # both joined

def worth(team):                                         # value of any team of players
    return v[frozenset(team)]                            # look it up in the table

def marginal(order, player):                             # what one player adds in one order
    before = order[:order.index(player)]                 # everyone who arrived earlier
    return worth(before + [player]) - worth(before)      # value after minus value before

orders = [[0, 1], [1, 0]]                                # both possible arrival orders
phi = np.zeros(2)                                        # one Shapley value per player

for player in (0, 1):                                    # handle each player in turn
    total = 0.0                                          # accumulator across orders
    for order in orders:                                 # average over both orders
        total = total + marginal(order, player)          # add this order's marginal
    phi[player] = total / len(orders)                    # divide by the number of orders
    print(f"player {player + 1}: phi = {phi[player]:.2f}")   # 0.60 then 0.40

print("sum of phi =", round(phi.sum(), 2))               # 1.00: no value lost
print("gap v(both) - v(empty) =", round(v_both - v_empty, 2))   # 1.00: the books balance

**Line by line**

- `v = {...}` — a dictionary from each coalition to its value.
- `worth(team)` — converts a list of players into the dictionary key.
- `marginal(order, player)` — value added when `player` joins at their turn.
- `order[:order.index(player)]` — the players already present.
- `orders = [[0, 1], [1, 0]]` — the two orders in which players can arrive.
- `phi = np.zeros(2)` — one slot per player for the final numbers.
- `total / len(orders)` — the average marginal is the Shapley value.
- `phi.sum()` — the efficiency check: the shares must add up to the whole.

## 3. Plot: two orders, one answer

The two solo values already add up to 1.0, so both arrival orders hand out the same marginals and the average is unambiguous. The chart shows the two orders side by side with the Shapley split next to them.

In [ ]:
import matplotlib.pyplot as plt                          # plotting library, preinstalled on Colab

m_first = [marginal([0, 1], 0), marginal([0, 1], 1)]     # order: feature 1, then feature 2
m_second = [marginal([1, 0], 0), marginal([1, 0], 1)]    # order: feature 2, then feature 1
idx = np.arange(2)                                       # bar positions for the two players
labels = ["feature 1", "feature 2"]                      # names on the x-axis

plt.figure(figsize=(6, 3))                               # wide enough for three bars per player
plt.bar(idx - 0.25, m_first, width=0.25, color="tab:blue", label="1 then 2")    # first order
plt.bar(idx, m_second, width=0.25, color="tab:orange", label="2 then 1")        # second order
plt.bar(idx + 0.25, phi, width=0.25, color="tab:green", label="average = phi")  # the average
plt.xticks(idx, labels)                                  # label the groups
plt.ylabel("marginal contribution")                      # y-axis label
plt.title("solo values add up, so every order agrees")    # the takeaway
plt.legend(fontsize=7)                                   # name the three bars
plt.show()                                               # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library.
- `m_first` — marginals when player 1 arrives first.
- `m_second` — marginals when player 2 arrives first.
- `idx = np.arange(2)` — two x positions, one per player.
- `plt.bar(...)` — three offset bars per player: order 1, order 2, average.
- `plt.xticks(...)` — replace positions with feature names.
- `plt.title(...)` — states the result the chart supports.

## 4. Interactive: change one player's solo value

Slide the value of feature 1 alone and watch both shares move. Once the solo values no longer add up to 1.0, the two arrival orders disagree and the average is what settles the split. The pair still sums to 1.0, because Shapley conserves the total.

In [ ]:
from ipywidgets import interact                          # sliders that rerun a function

def show(v1_now=0.6):                                    # v1_now comes from the slider
    v2_now = 0.4                                         # feature 2 alone stays fixed
    both = 1.0                                           # the team's value stays fixed
    p1 = 0.5 * v1_now + 0.5 * (both - v2_now)            # average of feature 1's two marginals
    p2 = 0.5 * v2_now + 0.5 * (both - v1_now)            # average of feature 2's two marginals

    plt.figure(figsize=(5, 3))                           # fresh figure for each drag
    plt.bar(["feature 1", "feature 2"], [p1, p2], color=["tab:blue", "tab:orange"])   # one bar per player's share
    plt.ylim(0, 1)                                       # shares live between 0 and 1
    plt.ylabel("Shapley share")                          # y-axis label
    plt.title(f"v({{1}}) = {v1_now:.2f} -> phi = {p1:.2f}, {p2:.2f} (sum {p1 + p2:.2f})")   # readout
    plt.show()                                           # draw it

interact(show, v1_now=(0.0, 1.0, 0.05))                  # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(v1_now=0.6)` — the function that redraws one frame.
- `p1 = 0.5 * ... + 0.5 * ...` — feature 1's two marginals, averaged.
- `p2 = ...` — same for feature 2, so the two marginals swap roles.
- `plt.bar(...)` — draw the current split.
- `plt.title(...)` — the live readout; double braces print a literal brace.
- `interact(show, v1_now=(...))` — the slider, its range and step.

## 5. When it breaks: LIME on four perturbations

LIME fits a simple model to a handful of random perturbations near the input. With only four perturbations, which feature looks most important depends on the seed. Same model, same input, different explanation.

In [ ]:
def black_box(mask):                                     # the model being explained: a weighted sum
    return 0.6 * mask[0] + 0.4 * mask[1]                 # feature 1 truly matters more

rows = []                                                # (seed, coef1, coef2) per run
flips = 0                                                # how often feature 2 wins

for seed in range(8):                                    # eight independent resamplings
    rng = np.random.default_rng(seed)                    # fresh random stream per seed
    masks = rng.integers(0, 2, size=(4, 2)).astype(float)   # four binary perturbations
    labels = np.array([black_box(m) for m in masks])     # model output for each
    dist = (1 - masks).sum(axis=1)                       # how far each perturbation sits from the input
    sw = np.sqrt(np.exp(-dist))                          # closeness weight, square-rooted for least squares
    A = np.c_[np.ones(4), masks] * sw[:, None]           # design matrix, rows weighted
    y_w = labels * sw                                    # targets, weighted the same way
    coef, *_ = np.linalg.lstsq(A, y_w, rcond=None)       # weighted linear surrogate
    flipped = abs(coef[1]) < abs(coef[2])                # did feature 2 outrank feature 1?
    flips = flips + int(flipped)                         # count the disagreement
    rows.append((seed, coef[1], coef[2]))                # keep the numbers for the plot
    print(f"seed {seed}: coef1 = {coef[1]:+.2f}, coef2 = {coef[2]:+.2f}, top = {'2' if flipped else '1'}")   # one row per seed

print("feature 2 ranked first in", flips, "of 8 runs")  # 2 of 8: unstable

plt.figure(figsize=(6, 3))                               # room for two bars per seed
pos = np.arange(len(rows))                               # one x position per seed
plt.bar(pos - 0.2, [r[1] for r in rows], width=0.4, color="tab:blue", label="coef for feature 1")    # feature 1
plt.bar(pos + 0.2, [r[2] for r in rows], width=0.4, color="tab:orange", label="coef for feature 2")  # feature 2
plt.axhline(0.6, color="tab:blue", linestyle="--", linewidth=0.9)     # the truth for feature 1
plt.axhline(0.4, color="tab:orange", linestyle="--", linewidth=0.9)   # the truth for feature 2
plt.xticks(pos, [f"s{r[0]}" for r in rows])              # label each seed
plt.ylabel("LIME coefficient")                           # y-axis label
plt.title("same model, fresh seed: the ranking flips")   # the takeaway
plt.legend(fontsize=7)                                   # name the two series
plt.show()                                               # render

**Line by line**

- `def black_box(mask)` — the model, written so we can query any perturbation.
- `rows = []` and `flips = 0` — storage and the disagreement counter.
- `for seed in range(8)` — eight independent runs of the same procedure.
- `rng.integers(0, 2, size=(4, 2))` — four random on/off perturbations of two features.
- `labels = ...` — ask the black box about each perturbation.
- `sw = ...` — LIME's closeness weight, square-rooted so least squares applies it once.
- `A = ...` and `y_w = ...` — weighted design matrix and targets.
- `coef, *_ = np.linalg.lstsq(...)` — fit the weighted linear surrogate.
- `flipped = ...` — did the surrogate reverse the true ranking?
- `plt.bar(...)` — coefficients per seed, with dashed lines at the truth.
- `plt.title(...)` — names the failure: the answer depends on the seed.

## Your turn

In section 5, change `size=(4, 2)` to `size=(20, 2)` and predict what happens to `flips` before you rerun. Then set the section 4 slider to `v({1}) = 0.4` and explain why the two shares become equal.